### Food Delivery Operations Analytics
**Goal:** Load the raw dataset, understand its structure, fix quality 
issues, create new columns, and save a clean version for analysis.

**Input:** `../data/raw/zomato-delivery-dataset.csv`  
**Output:** `../data/processed/orders_clean.csv`

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

Libraries loaded successfully
Pandas version: 3.0.2
Numpy version: 2.4.4


### Step 1: Load the Raw Data

In [23]:
df = pd.read_csv('../data/raw/zomato-delivery-dataset.csv')

print(f"Dataset shape: {df.shape}")
print(f"Total rows: {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")
print("\nColumn names (raw):")
print(df.columns.tolist())
print("\nFirst 3 rows:")
df.head(3)

Dataset shape: (45584, 20)
Total rows: 45,584
Total columns: 20

Column names (raw):
['ID', 'Delivery_person_ID', 'Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'Order_Date', 'Time_Orderd', 'Time_Order_picked', 'Weather_conditions', 'Road_traffic_density', 'Vehicle_condition', 'Type_of_order', 'Type_of_vehicle', 'multiple_deliveries', 'Festival', 'City', 'Time_taken (min)']

First 3 rows:


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken (min)
0,0xcdcd,DEHRES17DEL01,36.0,4.2,30.327968,78.046106,30.397968,78.116106,12-02-2022,21:55,22:10,Fog,Jam,2,Snack,motorcycle,3.0,No,Metropolitian,46
1,0xd987,KOCRES16DEL01,21.0,4.7,10.003064,76.307589,10.043064,76.347589,13-02-2022,14:55,15:05,Stormy,High,1,Meal,motorcycle,1.0,No,Metropolitian,23
2,0x2784,PUNERES13DEL03,23.0,4.7,18.562450,73.916619,18.652450,74.006619,04-03-2022,17:30,17:40,Sandstorms,Medium,1,Drinks,scooter,1.0,No,Metropolitian,21


### Step 2 — Rename Columns

- Renaming all 20 columns to lowercase, underscores instead of spaces.
- This makes the code cleaner and prevents errors from column name mismatches.
- We also strip leading/trailing spaces from column names.

In [24]:
df.columns = df.columns.str.strip()

df.rename(columns={
    'ID': 'order_id',
    'Delivery_person_ID': 'delivery_person_id',
    'Delivery_person_Age': 'delivery_person_age',
    'Delivery_person_Ratings': 'delivery_person_rating',
    'Restaurant_latitude': 'restaurant_lat',
    'Restaurant_longitude': 'restaurant_lon',
    'Delivery_location_latitude': 'delivery_lat',
    'Delivery_location_longitude': 'delivery_lon',
    'Order_Date': 'order_date',
    'Time_Orderd': 'time_ordered',
    'Time_Ordered': 'time_ordered',
    'Time_Order_picked': 'time_picked',
    'Weather_conditions': 'weather',
    'Road_traffic_density': 'traffic_density',
    'Vehicle_condition': 'vehicle_condition',
    'Type_of_order': 'order_type',
    'Type_of_vehicle': 'vehicle_type',
    'multiple_deliveries': 'multiple_deliveries',
    'Festival': 'festival',
    'City': 'city',
    'Time_taken(min)': 'time_taken_min',
    'Time_taken (min)': 'time_taken_min'
}, inplace=True)

required_cols = [
    'order_id', 'delivery_person_id', 'delivery_person_age',
    'delivery_person_rating', 'restaurant_lat', 'restaurant_lon',
    'delivery_lat', 'delivery_lon', 'order_date', 'time_ordered',
    'time_picked', 'weather', 'traffic_density', 'vehicle_condition',
    'order_type', 'vehicle_type', 'multiple_deliveries',
    'festival', 'city', 'time_taken_min'
]

missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise KeyError(f"Missing columns: {missing_cols}")

print("All required columns found.")
print("\nColumns after renaming:")
print(df.columns.tolist())

All required columns found.

Columns after renaming:
['order_id', 'delivery_person_id', 'delivery_person_age', 'delivery_person_rating', 'restaurant_lat', 'restaurant_lon', 'delivery_lat', 'delivery_lon', 'order_date', 'time_ordered', 'time_picked', 'weather', 'traffic_density', 'vehicle_condition', 'order_type', 'vehicle_type', 'multiple_deliveries', 'festival', 'city', 'time_taken_min']


### Step 3 — Replace Hidden String Nulls

- Some columns in this dataset store missing values as the *text* `"NaN"` — not as actual null values.
pandas cannot detect or fill these with `fillna()` — it sees them as valid text.

- We replace all string variants (`NaN`, `nan`, `NULL`, `''`, `' '`) with `np.nan` (real Python null)
so that the next steps can properly detect and handle them.

In [25]:
string_null_variants = [
    'NaN', 'nan', 'NAN',
    'NaT', 'nat', 'NAT',
    'NULL', 'null',
    'None', 'none',
    '', ' '
]

cols_to_clean = [
    'delivery_person_age', 'delivery_person_rating',
    'multiple_deliveries', 'festival', 'city',
    'weather', 'traffic_density',
    'time_ordered', 'time_picked', 'order_date'
]

for col in cols_to_clean:
    before = df[col].isnull().sum()
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace(string_null_variants, np.nan)
    after = df[col].isnull().sum()
    print(f"{col}: {before} nulls before → {after} nulls after string cleanup")

delivery_person_age: 1854 nulls before → 1854 nulls after string cleanup
delivery_person_rating: 1908 nulls before → 1908 nulls after string cleanup
multiple_deliveries: 993 nulls before → 993 nulls after string cleanup
festival: 228 nulls before → 228 nulls after string cleanup
city: 1200 nulls before → 1200 nulls after string cleanup
weather: 616 nulls before → 616 nulls after string cleanup
traffic_density: 601 nulls before → 601 nulls after string cleanup
time_ordered: 1731 nulls before → 1731 nulls after string cleanup
time_picked: 0 nulls before → 0 nulls after string cleanup
order_date: 0 nulls before → 0 nulls after string cleanup


### Step 4 — Fix Data Types

Converting each column to its correct type:
- Numeric columns: `pd.to_numeric(errors='coerce')` — bad values become NaN instead of crashing
- Date column: `pd.to_datetime()` — 52% of order_date will be NaT (expected, documented)
- Category columns: `.astype('category')` — saves memory, enables category operations
- Target column `time_taken_min`: strip text prefix like `"(min) 24"` → extract number only

In [26]:
df['delivery_person_age'] = pd.to_numeric(df['delivery_person_age'], errors='coerce')
df['delivery_person_rating'] = pd.to_numeric(df['delivery_person_rating'], errors='coerce')
df['multiple_deliveries'] = pd.to_numeric(df['multiple_deliveries'], errors='coerce')

try:
    df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', errors='coerce', dayfirst=True)
except TypeError:
    df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce', dayfirst=True)

print(f"order_date NaT after conversion: {df['order_date'].isna().sum():,} (expected ~52% of rows)")

df['time_taken_min'] = (
    df['time_taken_min']
    .astype(str)
    .str.extract(r'(\d+)')
    .astype(float)
)
print(f"time_taken_min — min: {df['time_taken_min'].min()}, max: {df['time_taken_min'].max()}")

category_cols = ['weather', 'traffic_density', 'order_type', 'vehicle_type', 'festival', 'city']
for col in category_cols:
    df[col] = df[col].astype('category')

print("\nData types after conversion:")
print(df.dtypes)

order_date NaT after conversion: 0 (expected ~52% of rows)
time_taken_min — min: 10.0, max: 54.0

Data types after conversion:
order_id                             str
delivery_person_id                   str
delivery_person_age              float64
delivery_person_rating           float64
restaurant_lat                   float64
restaurant_lon                   float64
delivery_lat                     float64
delivery_lon                     float64
order_date                datetime64[us]
time_ordered                         str
time_picked                          str
weather                         category
traffic_density                 category
vehicle_condition                  int64
order_type                      category
vehicle_type                    category
multiple_deliveries              float64
festival                        category
city                            category
time_taken_min                   float64
dtype: object


### Step 5 — Handle Missing Values

**Drop:** Rows where `time_ordered`, `time_picked`, or `time_taken_min` is null.
These are the core time columns. Without them we cannot compute any time features. Only ~0.3% of rows affected.

**Fill strategy per column:**
| Column | Strategy | Reason |
|---|---|---|
| delivery_person_age | Median | Numeric — robust to outliers |
| delivery_person_rating | Median | Numeric — robust to outliers |
| multiple_deliveries | 0 | Missing = single delivery |
| festival, city, weather, traffic_density | Mode | Categorical — use most common value |
| order_date | Keep NaT | 52% missing — cannot impute dates |

In [27]:
rows_before = len(df)
df.dropna(subset=['time_ordered', 'time_picked', 'time_taken_min'], inplace=True)
rows_after = len(df)
print(f"Dropped {rows_before - rows_after} rows with missing time values ({(rows_before-rows_after)/rows_before*100:.2f}%)")

age_median = df['delivery_person_age'].median()
df['delivery_person_age'] = df['delivery_person_age'].fillna(age_median)
df['delivery_person_age'] = df['delivery_person_age'].round(0).astype('Int64')
print(f"delivery_person_age → filled with median {age_median}")

rating_median = df['delivery_person_rating'].median()
df['delivery_person_rating'] = df['delivery_person_rating'].fillna(rating_median)
print(f"delivery_person_rating → filled with median {rating_median:.2f}")

df['multiple_deliveries'] = df['multiple_deliveries'].fillna(0)
df['multiple_deliveries'] = df['multiple_deliveries'].round(0).astype('Int64')
print("multiple_deliveries → filled with 0")

for col in ['festival', 'city', 'weather', 'traffic_density']:
    mode_val = df[col].mode(dropna=True)[0]
    df[col] = df[col].fillna(mode_val)
    print(f"{col} → filled with mode '{mode_val}'")

cols_to_check = [c for c in df.columns if c != 'order_date']
remaining_nulls = df[cols_to_check].isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]

print("\n=== NULL CHECK (excluding order_date) ===")
if len(remaining_nulls) == 0:
    print("Zero unexpected nulls. All columns clean.")
else:
    print("WARNING — unexpected nulls remain:")
    print(remaining_nulls)

print(f"\norder_date NaT (intentional): {df['order_date'].isnull().sum():,}")

Dropped 1731 rows with missing time values (3.80%)
delivery_person_age → filled with median 30.0
delivery_person_rating → filled with median 4.70
multiple_deliveries → filled with 0
festival → filled with mode 'No'
city → filled with mode 'Metropolitian'
weather → filled with mode 'Fog'
traffic_density → filled with mode 'Low'

=== NULL CHECK (excluding order_date) ===
Zero unexpected nulls. All columns clean.

order_date NaT (intentional): 0


### Step 6 — Feature Engineering

Creating new columns that power the entire analysis:
- `sla_breached` — our primary KPI flag (delivery > 30 min = True)
- `order_hour` — extracted from time string for peak-hour analysis
- `order_day_of_week`, `is_weekend`, `order_month` — from order_date (only ~48% of rows will have these)
- `distance_km` — Haversine great-circle distance between restaurant and customer GPS
- `delivery_speed` — categorical speed bucket (Fast / On-time / Late / Very Late)

In [ ]:
def fix_order_time(val):
    val = str(val).strip()
    try:
        f = float(val)
        if 0.0 <= f < 1.0:
            total_min = round(f * 24 * 60)
            h = (total_min // 60) % 24
            m = total_min % 60
            return f"{h:02d}:{m:02d}"
    except ValueError:
        pass

    # wrap hours >= 24 with % 24
    if ':' in val:
        parts = val.split(':')
        try:
            h = int(parts[0]) % 24      # 24→0, 25→1, keeps 0–23 unchanged
            m = int(parts[1])
            return f"{h:02d}:{m:02d}"
        except (ValueError, IndexError):
            pass

    # anything unparseable → default to midnight
    return "00:00"


df['time_ordered'] = df['time_ordered'].apply(fix_order_time)
df['time_picked']  = df['time_picked'].apply(fix_order_time)

df['order_hour'] = (
    df['time_ordered']
    .str.extract(r'^(\d{2})')
    .squeeze()
    .astype(int)
)
print(f"order_hour nulls: {df['order_hour'].isnull().sum()}")

df['order_day_of_week'] = df['order_date'].dt.day_name()
df['is_weekend'] = df['order_date'].dt.dayofweek >= 5
df['order_month'] = df['order_date'].dt.month

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

df['distance_km'] = haversine(
    df['restaurant_lat'], df['restaurant_lon'],
    df['delivery_lat'], df['delivery_lon']
)
print(f"distance_km — min: {df['distance_km'].min():.1f}, max: {df['distance_km'].max():.1f}, mean: {df['distance_km'].mean():.1f}")

def speed_category(mins):
    if mins <= 20: return 'Fast'
    elif mins <= 30: return 'On-time'
    elif mins <= 45: return 'Late'
    else: return 'Very Late'

df['delivery_speed'] = df['time_taken_min'].apply(speed_category)

# SLA breached column
df['sla_breached'] = df['time_taken_min'] > 30

print("\nDelivery speed distribution:")
print(df['delivery_speed'].value_counts())

order_hour nulls: 0
distance_km — min: 1.5, max: 6884.7, mean: 27.2

Delivery speed distribution:
delivery_speed
On-time      17170
Fast         13603
Late         11591
Very Late     1489
Name: count, dtype: int64


### Step 7 — Flag Outliers and Save Clean Dataset

In [29]:
# outlier detection (business-rule method)
print("=== time_taken_min outlier check ===")
print(f"Min:    {df['time_taken_min'].min()} minutes")
print(f"Max:    {df['time_taken_min'].max()} minutes")
print(f"Mean:   {df['time_taken_min'].mean():.1f} minutes")
print(f"Median: {df['time_taken_min'].median()} minutes")

too_fast = df[df['time_taken_min'] < 5]
too_slow = df[df['time_taken_min'] > 120]
print(f"\nOrders under 5 min (likely data errors):  {len(too_fast)}")
print(f"Orders over 120 min (extreme outliers):   {len(too_slow)}")

df['is_outlier'] = (df['time_taken_min'] < 5) | (df['time_taken_min'] > 120)
print(f"\nTotal flagged as outliers: {df['is_outlier'].sum()}")
print("Note: Outliers are flagged, not deleted. Analyst decision retained.")

# final validation
print("\n=== FINAL VALIDATION ===")
print(f"Final shape: {df.shape}")
print(f"\nAll columns:")
print(df.columns.tolist())

cols_to_check = [c for c in df.columns if c != 'order_date']
remaining_nulls = df[cols_to_check].isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]

if len(remaining_nulls) == 0:
    print("\nZero unexpected nulls. Data is clean.")
else:
    print("\nWARNING — unexpected nulls:")
    print(remaining_nulls)

print(f"\norder_date NaT (intentional, documented): {df['order_date'].isnull().sum():,}")
print(f"\nFinal dtypes:")
print(df.dtypes)

# save clean dataset
df.to_csv('../data/processed/orders_clean.csv', index=False)
print("\nSaved → data/processed/orders_clean.csv")

=== time_taken_min outlier check ===
Min:    10.0 minutes
Max:    54.0 minutes
Mean:   26.3 minutes
Median: 26.0 minutes

Orders under 5 min (likely data errors):  0
Orders over 120 min (extreme outliers):   0

Total flagged as outliers: 0
Note: Outliers are flagged, not deleted. Analyst decision retained.

=== FINAL VALIDATION ===
Final shape: (43853, 28)

All columns:
['order_id', 'delivery_person_id', 'delivery_person_age', 'delivery_person_rating', 'restaurant_lat', 'restaurant_lon', 'delivery_lat', 'delivery_lon', 'order_date', 'time_ordered', 'time_picked', 'weather', 'traffic_density', 'vehicle_condition', 'order_type', 'vehicle_type', 'multiple_deliveries', 'festival', 'city', 'time_taken_min', 'order_hour', 'order_day_of_week', 'is_weekend', 'order_month', 'distance_km', 'delivery_speed', 'sla_breached', 'is_outlier']

Zero unexpected nulls. Data is clean.

order_date NaT (intentional, documented): 0

Final dtypes:
order_id                             str
delivery_person_id   